# Joint Archetype Deconvolution over PermCell Signature Scores

The fourth and last configuration in this series. The same 354,435 cells have now been deconvolved
four ways:

| Analysis | Space | Fitting | Notebook |
|---|---|---|---|
| 1 | 29 markers | joint, offset arms | `CellLines_CyEmbed_JointAnalysis` |
| 2 | 29 markers | per line, matched after | `CellLines_CyEmbed_PerLine_Matching` |
| 3 | 15 signatures | per line, matched after | `CellLines_PermCell_Archetypes` |
| **4** | **15 signatures** | **joint, offset arms** | **this notebook** |

This one completes the 2×2: marker vs signature space × joint vs per-line fitting. Everything that
differs from analysis 1 is the representation; everything that differs from analysis 3 is the
fitting strategy. So disagreements are attributable.

### Why the joint fit is possible here, and what it buys

With all five lines pooled, `use_sample_offset` becomes identifiable again (5 samples rather than 1),
so both arms run as in analysis 1:

* **Arm A** — `use_sample_offset=False`
* **Arm B** — `use_sample_offset=True`, per-line intercepts **in signature space**

Arm B is the interesting one. PermCell already row-centres each cell, which removes the global
chromatin-magnitude axis that consumed two of nine archetypes in analysis 1. If Arm B still beats
Arm A, that means there are genuine per-line *programme* baselines (some lines simply run more
luminal, or more Polycomb-repressed) over and above the magnitude effect.

### The scores are reused, and that is provably safe

`sipsic_like_scores_v3` is a **strictly per-cell** statistic — verified below by scoring the same
cells in five different contexts. So joint scoring is identical to concatenating the per-line score
files, and the saved scores are reused directly.

## 0. Setup

In [ ]:
import sys, json, itertools
from pathlib import Path

CE_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Experiments/CyEmbed"
HELPER = "/Users/ronguy/Dropbox/Work/CyTOF/HelperPackage"
CT_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Code/cytof-transform"
SCRIPTS = "/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/scripts"
for p in (CE_PATH, HELPER, CT_PATH, SCRIPTS):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np, pandas as pd, anndata as ad
import matplotlib.pyplot as plt, seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

from CyEmbed.data import extract_matrix, fit_scaler, preprocess_array, split_train_val_indices
from CyEmbed.train import build_sweep_configs, run_sweep
from CyEmbed.analysis import load_run_outputs
from permcell_signatures import SIGNATURES

plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 10, "ytick.labelsize": 10})

BASE = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
PLOTS = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)
SCORES = BASE / "outputs/permcell_scores"
JSIG = BASE / "outputs/cyembed_joint_signature_sweep"
JMARK = BASE / "outputs/cyembed_joint_sweep"
PERLINE_SIG = BASE / "outputs/cyembed_signature_sweep"

LINES = ["MDAMB468", "HCC70", "SUM149", "HCC1937", "MCF7"]
LINE_DISP = {"MDAMB468": "MDA-MB-468", "HCC70": "HCC70", "SUM149": "SUM149",
             "HCC1937": "HCC1937", "MCF7": "MCF7"}
LINE_ORDER = ["HCC1937", "HCC70", "MCF7", "MDA-MB-468", "SUM149"]
ARM_PALETTE = {False: "#e41a1c", True: "#377eb8"}

SEED = 42
K_RANGE = list(range(4, 15))
SEEDS = [42, 1, 2]
FAILURE_TOLERANCE = 0.05

BASE_CONFIG = dict(
    model_type="deterministic", decoder_type="factorized",
    d=16, hidden_dims=[64, 32], tau=1.0,
    epochs=1500, early_stopping=True, patience=20,
    min_delta=0.0, restore_best_weights=True,
    lr=1e-3, batch_size=2048, weight_decay=1e-5, dropout=0.0,
    logit_normalizer="entmax", entmax_alpha=1.5, grad_clip_norm=5.0,
    separation_mode="cosine_sq", balance_mode="l2_uniform",
    lambda_entropy=1e-3, lambda_sep=1e-3, lambda_balance=0.05,
    recon_loss_type="mse", device="cpu", deterministic=True,
    seed=SEED, n_samples=len(LINES),
)
print(f"{len(SIGNATURES)} discovery signatures")

## 1. Joint dataset, and proof that reusing per-line scores is valid

In [ ]:
zs = {l: pd.read_csv(SCORES / f"{l}_Z.csv") for l in LINES}
SIG_NAMES = list(zs[LINES[0]].columns)
X_raw = np.vstack([zs[l].to_numpy(np.float32) for l in LINES])
sample_ids_all = np.concatenate([np.repeat(LINE_DISP[l], len(zs[l])) for l in LINES])

joint = ad.AnnData(X=X_raw,
                   obs=pd.DataFrame({"cell_line": sample_ids_all},
                                    index=[f"c{i}" for i in range(len(X_raw))]),
                   var=pd.DataFrame(index=SIG_NAMES))
bundle = extract_matrix(adata=joint, layer=None, sample_col="cell_line")
SCALER, _ = fit_scaler(bundle.X, mode="zscore",
                       sample_ids=bundle.sample_ids, balanced_max_per_sample=5000)
X_scaled = preprocess_array(bundle.X, SCALER)
train_idx, val_idx = split_train_val_indices(n_cells=len(X_scaled), val_fraction=0.2,
                                             seed=SEED, stratify_labels=bundle.sample_ids)
print(f"joint: {X_scaled.shape[0]:,} cells x {X_scaled.shape[1]} signatures")
print(f"train {len(train_idx):,} / val {len(val_idx):,}")
print(pd.Series(sample_ids_all).value_counts().to_string())

In [ ]:
# --- 1.1 Is sipsic_like_scores_v3 really a PER-CELL statistic? ---------------------------------
# Methods like Seurat AddModuleScore / scanpy score_genes pick control genes from DATASET-average
# expression, so their scores move when the cell set changes. PermCell row-centres each cell and
# builds its null by permuting markers WITHIN that cell. Tested directly: score the same 500 MCF7
# cells in five different contexts and compare.
import PermCell_Smooth as PCS
from cytofstandard import Project

marker_ads = {}
for line in LINES:
    a = Project.load(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{line}_NormCompare") \
               .get_run(line).read_adata()
    bio = [m for m in a.var_names if m not in ["H3", "H3.3", "H4"]]
    sub = ad.AnnData(X=a[:, bio].layers["norm_divide"].copy(),
                     obs=a.obs[["cell_uuid"]].copy(), var=a[:, bio].var.copy())
    sub.obs["cell_line"] = LINE_DISP[line]
    marker_ads[line] = sub
mk_comb = ad.concat([marker_ads[l] for l in LINES], join="outer")
mk_b = extract_matrix(adata=mk_comb, layer=None, sample_col="cell_line")
MK_SCALER, _ = fit_scaler(mk_b.X, mode="zscore",
                          sample_ids=mk_b.sample_ids, balanced_max_per_sample=5000)
X_marker_all = preprocess_array(mk_b.X, MK_SCALER).astype(np.float32)
MARKERS = list(mk_comb.var_names)
VAR_MK = mk_comb.var.copy()


def _score(Xs):
    t = ad.AnnData(X=np.ascontiguousarray(Xs),
                   obs=pd.DataFrame(index=[f"c{i}" for i in range(len(Xs))]), var=VAR_MK.copy())
    Z, _, _, _, _ = PCS.sipsic_like_scores_v3(
        adata=t, marker_sets=SIGNATURES, layer="X", n_perm=2000, seed=0, exclude_set=True,
        two_sided=True, abs_variant=True, progress=False, normalize_set_weights="l2",
        prefer_permutation=True, perm_batch=512)
    return Z.values


rng_t = np.random.default_rng(7)
tgt = rng_t.choice(np.flatnonzero(sample_ids_all == "MCF7"), 500, replace=False)
Xt = X_marker_all[tgt]
sm = rng_t.choice(np.flatnonzero(sample_ids_all == "SUM149"), 50000, replace=False)
perm = rng_t.permutation(500)

contexts = {
    "alone (500 MCF7)": _score(Xt),
    "+50k SUM149 appended": _score(np.vstack([Xt, X_marker_all[sm]]))[:500],
    "+50k SUM149 prepended": _score(np.vstack([X_marker_all[sm], Xt]))[50000:],
    "row order shuffled": _score(Xt[perm])[np.argsort(perm)],
}
ref = contexts["alone (500 MCF7)"]
print("Same 500 MCF7 cells scored in different contexts:\n")
print(f"{'context':<28}{'max |diff|':>14}{'corr':>10}")
for k, v in contexts.items():
    print(f"{k:<28}{np.abs(v - ref).max():>14.3e}{np.corrcoef(v.ravel(), ref.ravel())[0, 1]:>10.6f}")
worst = max(np.abs(v - ref).max() for v in contexts.values())
print(f"\nmax deviation = {worst:.3e} -> "
      + ("PER-CELL confirmed: joint scoring == concatenating per-line scores."
         if worst < 1e-6 else "cross-cell dependency detected -- rescore jointly!"))
print("\nNOTE: this holds for the SCORING function only. PermCell's Gaussian smoothing step IS")
print("cross-cell by construction, which is a further reason to use the raw scores.")

## 2. Joint sweep: Arm A (no offset) vs Arm B (per-line offsets in signature space)

K = 4..14 × seeds {42, 1, 2} × both arms = 66 runs. Cached by fingerprint.
Equivalent script: `scripts/cyembed_joint_signature_sweep.py`.

In [ ]:
stale = [d for d in JSIG.glob("run_*") if d.is_dir() and not any(d.iterdir())]
for d in stale:
    d.rmdir()

configs = build_sweep_configs({"K": K_RANGE, "use_sample_offset": [False, True], "seed": SEEDS})
_ = run_sweep(x=X_scaled, marker_names=list(bundle.marker_names),
              cell_ids=list(bundle.cell_ids), output_root=JSIG,
              base_config=BASE_CONFIG, sweep_configs=configs,
              train_idx=train_idx, val_idx=val_idx, sample_ids=bundle.sample_ids,
              scaler_state=SCALER.to_dict())

rows = []
for r in sorted(JSIG.glob("run_*")):
    sf, cf = r / "summary_metrics.json", r / "config.json"
    if not (sf.exists() and cf.exists()):
        continue
    sm, cfg = json.loads(sf.read_text()), json.loads(cf.read_text())
    rows.append({"run_id": r.name, "run_dir": str(r), "K": cfg["K"],
                 "use_sample_offset": cfg.get("use_sample_offset", False),
                 "seed": cfg["seed"],
                 "val_recon": sm.get("best_val_recon", sm.get("val", {}).get("recon_mse"))})
runs_df = pd.DataFrame(rows)

want = {(k, o, s) for k, o, s in itertools.product(K_RANGE, [False, True], SEEDS)}
have = {(r.K, r.use_sample_offset, r.seed) for r in runs_df.itertuples()}
missing = sorted(want - have)
print(f"runs: {len(runs_df)}/{len(want)}")
print("grid complete" if not missing else f"!! MISSING {len(missing)}: {missing[:8]}")

## 3. Evaluation: arms and cell-line mixing

Mixing uses the corrected null established in analysis 1 — under perfect mixing $p(s|k)$ equals each
line's **share of cells**, not a uniform $1/5$, so the ceiling is $H_\text{null}\approx2.296$ bits,
not $\log_2 5=2.322$. Both the soft-weight and the argmax view are reported, because the soft view
saturates whenever simplex weights are diffuse.

In [ ]:
def shannon_bits(p):
    p = np.asarray(p, float); p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def mixing(w, sids, order=LINE_ORDER):
    cols = [f"A{k+1}" for k in range(w.shape[1])]
    dfw = pd.DataFrame(w, columns=cols); dfw["l"] = np.asarray(sids)
    ps = dfw.groupby("l")[cols].sum().reindex(order); ps = ps / ps.sum(0)
    dom = pd.Series([f"A{k+1}" for k in w.argmax(1)])
    ct = pd.crosstab(pd.Series(np.asarray(sids)), dom).reindex(order).reindex(columns=cols, fill_value=0)
    ph = ct / ct.sum(0).replace(0, np.nan)
    hn = shannon_bits(pd.Series(np.asarray(sids)).value_counts(normalize=True).reindex(order).values)
    hs = np.array([shannon_bits(ps[c].values) for c in cols])
    hh = np.array([shannon_bits(ph[c].fillna(0).values) for c in cols])
    return dict(h_null=hn, ratio_soft=hs.mean() / hn, ratio_hard=hh.mean() / hn,
                h_soft=hs, h_hard=hh, props_soft=ps, props_hard=ph, counts=ct)


H_NULL = shannon_bits(pd.Series(sample_ids_all).value_counts(normalize=True).reindex(LINE_ORDER).values)
print(f"count-based null = {H_NULL:.4f} bits   (log2(5) = {np.log2(5):.4f}, unattainable)\n")

ev = []
for _, r in runs_df.iterrows():
    w = np.load(Path(r["run_dir"]) / "W.npy")
    m = mixing(w, sample_ids_all)
    ev.append({"K": r["K"], "use_sample_offset": r["use_sample_offset"], "seed": r["seed"],
               "val_recon": r["val_recon"], "ratio_soft": m["ratio_soft"],
               "ratio_hard": m["ratio_hard"],
               "mean_max_w": float(w.max(1).mean()),
               "min_dom_share": float((m["counts"].sum(0) / m["counts"].to_numpy().sum()).min()),
               "run_dir": r["run_dir"]})
eval_df = pd.DataFrame(ev)

agg = (eval_df.groupby(["use_sample_offset", "K"])
       .agg(val_best=("val_recon", "min"), val_mean=("val_recon", "mean"),
            ratio_soft=("ratio_soft", "mean"), ratio_hard=("ratio_hard", "mean"),
            mean_max_w=("mean_max_w", "mean")).reset_index())
print("=== Arm comparison (best of 3 seeds) ===")
piv = agg.pivot(index="K", columns="use_sample_offset", values="val_best")
piv.columns = ["Arm A (no offset)", "Arm B (offset)"]
piv["gain_%"] = 100 * (1 - piv["Arm B (offset)"] / piv["Arm A (no offset)"])
print(piv.round(4).to_string())
print(f"\nArm B lower at {(piv['gain_%'] > 0).sum()}/{len(piv)} K values, "
      f"median gain {piv['gain_%'].median():.1f}%")
print("\n-> per-line offsets still help AFTER PermCell has row-centred every cell, so there are")
print("   genuine per-line PROGRAMME baselines, not merely a global magnitude difference.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.lineplot(data=eval_df, x="K", y="val_recon", hue="use_sample_offset", marker="o",
             ax=axes[0], palette=ARM_PALETTE, errorbar="sd")
axes[0].set_title("Validation reconstruction vs K\n(signature space, joint fit)", fontweight="bold")
axes[0].set_ylabel("val recon"); axes[0].grid(True, ls="--", alpha=0.5)

sns.lineplot(data=eval_df, x="K", y="ratio_hard", hue="use_sample_offset", marker="D",
             ax=axes[1], palette=ARM_PALETTE, errorbar="sd")
axes[1].axhline(1.0, color="black", lw=1.4, label="perfect mixing")
axes[1].set_title("Hard-assignment mixing ratio H/H_null\n(higher = more line-shared archetypes)",
                  fontweight="bold")
axes[1].set_ylabel("H_hard / H_null"); axes[1].legend(fontsize=9); axes[1].grid(True, ls="--", alpha=0.5)

sns.lineplot(data=eval_df, x="K", y="mean_max_w", hue="use_sample_offset", marker="o",
             ax=axes[2], palette=ARM_PALETTE, errorbar="sd")
axes[2].plot(sorted(eval_df.K.unique()), 1 / np.array(sorted(eval_df.K.unique())),
             ls="--", color="grey", label="uniform floor 1/K")
axes[2].set_title("Simplex weight concentration", fontweight="bold")
axes[2].set_ylabel("mean max weight"); axes[2].legend(fontsize=9); axes[2].grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(PLOTS / "PermCellJoint_Arms.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Model selection

Same rule as everywhere else in this series: seeds >5% above the best seed at a given $K$ are
optimiser failures; Kneedle elbow on the best-of-seeds curve; selected $K$ = largest $K$ at or below
the elbow where every seed converged. Cross-seed stability reported as a diagnostic.

In [ ]:
def kneedle_elbow(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    xn = (x - x.min()) / (np.ptp(x) + 1e-12); yn = (y - y.min()) / (np.ptp(y) + 1e-12)
    chord = yn[0] + (xn - xn[0]) * (yn[-1] - yn[0]) / (xn[-1] - xn[0] + 1e-12)
    return x[int(np.argmax(chord - yn))]


def matched_cosine(a1, a2):
    n1 = a1 / (np.linalg.norm(a1, axis=1, keepdims=True) + 1e-12)
    n2 = a2 / (np.linalg.norm(a2, axis=1, keepdims=True) + 1e-12)
    sim = n1 @ n2.T
    r, c = linear_sum_assignment(-sim)
    return float(sim[r, c].mean())


arm_b = eval_df[eval_df.use_sample_offset].copy()
arm_b["best_at_K"] = arm_b.groupby("K")["val_recon"].transform("min")
arm_b["excess"] = arm_b["val_recon"] / arm_b["best_at_K"] - 1
arm_b["converged"] = arm_b["excess"] <= FAILURE_TOLERANCE

fails = arm_b[~arm_b.converged]
print("=== Convergence failures (Arm B) ===")
if len(fails) == 0:
    print("  none")
else:
    for _, r in fails.sort_values(["K", "seed"]).iterrows():
        print(f"  K={int(r.K):<3} seed={int(r.seed):<3} {r.val_recon:.5f} vs "
              f"best {r.best_at_K:.5f} (+{100*r.excess:.1f}%)")

sel = []
for k, g in arm_b.groupby("K"):
    A = {int(r.seed): np.load(Path(r.run_dir) / "A_hat.npy") for r in g.itertuples()}
    sd = sorted(A)
    sims = [matched_cosine(A[i], A[j]) for ii, i in enumerate(sd) for j in sd[ii + 1:]]
    sel.append({"K": k, "n_seeds": len(g), "n_conv": int(g.converged.sum()),
                "val_best": g.val_recon.min(), "stability": float(np.mean(sims)),
                "ratio_hard": g.ratio_hard.mean(), "mean_max_w": g.mean_max_w.mean(),
                "_A": A, "_g": g})
sel_df = pd.DataFrame(sel).sort_values("K").reset_index(drop=True)

K_ELBOW = kneedle_elbow(sel_df.K.values, sel_df.val_best.values)
ok = sel_df[(sel_df.K <= K_ELBOW) & (sel_df.n_conv == sel_df.n_seeds)]
K_SEL = int(ok.K.max()) if len(ok) else int(sel_df.loc[sel_df.val_best.idxmin(), "K"])
print("\n=== Arm B selection diagnostics ===")
print(sel_df.drop(columns=["_A", "_g"]).round(4).to_string(index=False))
print(f"\nelbow K = {K_ELBOW} | all-seeds-converged at/below elbow: {sorted(ok.K.tolist())}")
print(f"SELECTED K = {K_SEL}")

s_ = sel_df[sel_df.K == K_SEL].iloc[0]
A_by_seed, g_ = s_["_A"], s_["_g"]
conv = set(g_.loc[g_.converged, "seed"].astype(int))
cand = [x for x in sorted(A_by_seed) if x in conv] or sorted(A_by_seed)
msim = {x: np.mean([matched_cosine(A_by_seed[x], A_by_seed[o]) for o in cand if o != x])
        for x in cand} if len(cand) > 1 else {cand[0]: np.nan}
MEDOID = max(msim, key=msim.get)
best_dir = g_.loc[g_.seed == MEDOID, "run_dir"].iloc[0]
best = load_run_outputs(best_dir)
A_hat, W = best["A_hat"], best["W"]
K_opt = A_hat.shape[0]
mix = mixing(W, sample_ids_all)
print(f"\nrepresentative run: seed {MEDOID} (mean matched cosine {msim[MEDOID]:.3f}) | {Path(best_dir).name}")
print(f"K={K_opt}  val_recon={float(g_.loc[g_.seed==MEDOID,'val_recon'].iloc[0]):.5f}")
print(f"mixing: soft {100*mix['ratio_soft']:.1f}% of null | hard {100*mix['ratio_hard']:.1f}% of null")

## 5. Identifying the joint archetypes

Each archetype is described by its signature profile (what was fitted), the mean observed **marker**
profile of the cells it dominates (the grounding check), and its cell-line composition.

In [ ]:
X_marker_joint = X_marker_all   # same cell order as X_scaled, verified by construction
assert X_marker_joint.shape[0] == W.shape[0]

dom = W.argmax(1)
sig_prof = pd.DataFrame(A_hat, columns=SIG_NAMES,
                        index=[f"A{k+1}" for k in range(K_opt)])
# centre on the dataset centroid so profiles read as deviations
sig_cen = sig_prof - X_scaled.mean(0)

rows = []
for k in range(K_opt):
    m = dom == k
    n = int(m.sum())
    sp = sig_cen.iloc[k]
    mp = pd.Series(X_marker_joint[m].mean(0), index=MARKERS) if n else pd.Series(np.nan, index=MARKERS)
    comp = pd.Series(sample_ids_all[m]).value_counts(normalize=True).reindex(LINE_ORDER).fillna(0)
    rows.append({
        "arch": f"A{k+1}", "n_cells": n, "pct": 100 * n / len(dom),
        "sig_up": "; ".join(f"{s}{sp[s]:+.1f}" for s in sp.sort_values(ascending=False).index[:3]),
        "sig_dn": "; ".join(f"{s}{sp[s]:+.1f}" for s in sp.sort_values().index[:3]),
        "mk_up": "; ".join(f"{s}{mp[s]:+.2f}" for s in mp.sort_values(ascending=False).index[:4]),
        "mk_dn": "; ".join(f"{s}{mp[s]:+.2f}" for s in mp.sort_values().index[:4]),
        "KI67": mp["KI67"], "top_line": comp.idxmax(), "top_line_pct": 100 * comp.max(),
        "H_hard": mix["h_hard"][k], "ratio_hard": mix["h_hard"][k] / H_NULL,
        "_sig": sp, "_mk": mp,
    })
ident = pd.DataFrame(rows)


def label_arch(sp):
    parts = []
    if sp["Proliferation"] > 1.5 and sp["DNA_damage_response"] < 0: parts.append("cycling")
    if sp["DNA_damage_response"] > 1.5 and sp["Proliferation"] < 0: parts.append("damage-arrested")
    if sp["EMT"] > 1.5 or sp["Basal_B_claudin_low"] > 1.5: parts.append("mesenchymal")
    if sp["Luminal"] > 1.5: parts.append("luminal")
    if sp["Basal_A"] > 1.5: parts.append("basal-A")
    if sp["Stem_CD44p_CD24n"] > 1.5: parts.append("stem-like")
    if sp["Polycomb_repression"] > 1.5 or sp["Bivalent_poised"] > 1.5: parts.append("Polycomb/bivalent")
    if sp["Constitutive_heterochromatin"] > 1.5: parts.append("heterochromatic")
    if sp["Enhancer_primed"] > 1.5: parts.append("enhancer-primed")
    if sp["H3K4_promoter_vs_enhancer"] > 1.5: parts.append("promoter-weighted")
    if sp["Open_chromatin"] > 1.5: parts.append("open-chromatin")
    if sp["Elongation"] > 1.5: parts.append("elongation-high")
    return " / ".join(dict.fromkeys(parts)) or f"{sp.idxmax()}-driven"


ident["label"] = [label_arch(r["_sig"]) for _, r in ident.iterrows()]
max_share = 100 * pd.Series(sample_ids_all).value_counts(normalize=True).max()
ident["verdict"] = np.where(ident["top_line_pct"] > 1.5 * max_share, "line-preferential", "shared")

print("=" * 100)
print(f"JOINT SIGNATURE-SPACE ARCHETYPES  (K={K_opt}, offset-corrected, seed {MEDOID})")
print("=" * 100)
for r in ident.itertuples():
    print(f"\n{r.arch}  {r.n_cells:,} cells ({r.pct:.1f}%)  [{r.verdict}]  "
          f"top line {r.top_line} {r.top_line_pct:.0f}%  mixing {r.ratio_hard:.2f}")
    print(f"    LABEL       : {r.label}")
    print(f"    signatures  : UP {r.sig_up}   |   DN {r.sig_dn}")
    print(f"    markers     : UP {r.mk_up}")
    print(f"                  DN {r.mk_dn}")

n_shared = int((ident.verdict == "shared").sum())
print(f"\n-> {n_shared}/{K_opt} archetypes shared across lines; "
      f"{K_opt - n_shared}/{K_opt} line-preferential")
ident.drop(columns=["_sig", "_mk"]).to_csv(
    BASE / "outputs/permcell_scores/joint_signature_archetypes.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, max(4, K_opt * 0.55)),
                         gridspec_kw={"width_ratios": [1.5, 1.5, 0.9]})
v = float(np.abs(sig_cen.values).max())
sns.heatmap(sig_cen, cmap="RdBu_r", center=0, vmin=-v, vmax=v, annot=True, fmt=".1f",
            annot_kws={"size": 7}, cbar_kws={"label": "centred signature score"}, ax=axes[0])
axes[0].set_title(f"Joint archetypes in SIGNATURE space (K={K_opt})", fontweight="bold")
axes[0].set_yticklabels([f"{r.arch}: {r.label[:34]}" for r in ident.itertuples()], rotation=0, fontsize=8)

mk_prof = pd.DataFrame([r["_mk"] for _, r in ident.iterrows()],
                       index=[r.arch for r in ident.itertuples()])
vm = float(np.abs(mk_prof.values).max())
sns.heatmap(mk_prof, cmap="RdBu_r", center=0, vmin=-vm, vmax=vm,
            cbar_kws={"label": "mean observed marker z"}, ax=axes[1])
axes[1].set_title("Same archetypes, MARKER grounding\n(mean observed z of dominated cells)",
                  fontweight="bold")
axes[1].tick_params(axis="x", labelsize=7)

comp = pd.crosstab(pd.Series(sample_ids_all), pd.Series([f"A{k+1}" for k in dom]))
comp = (comp / comp.sum(0) * 100).reindex(LINE_ORDER).T
comp.plot(kind="barh", stacked=True, ax=axes[2], colormap="Set2", edgecolor="black", linewidth=0.4)
axes[2].axvline(100 - max_share, color="black", ls=":", lw=1)
axes[2].set_title("Cell-line composition p(s|k)%", fontweight="bold")
axes[2].set_xlabel("% of archetype"); axes[2].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
axes[2].invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS / "PermCellJoint_Archetypes.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. The 2×2: how do all four analyses compare?

The headline question for this series: does representation (markers vs signatures) or fitting
strategy (joint vs per-line) matter more for finding **shared** biology?

In [ ]:
# joint MARKER-space reference model (analysis 1)
jm = load_run_outputs(JMARK / "run_deterministic_37e350d66f")
Wjm = jm["W"]
jm_sids = pd.read_csv(JMARK / "run_deterministic_37e350d66f" / "sample_ids.csv")["sample_id"].to_numpy()
mix_jm = mixing(Wjm, jm_sids)

summary_2x2 = pd.DataFrame([
    {"analysis": "1. joint / markers", "space": "29 markers", "fit": "joint",
     "K": Wjm.shape[1], "ratio_soft": mix_jm["ratio_soft"], "ratio_hard": mix_jm["ratio_hard"],
     "shared_frac": np.mean(mix_jm["h_hard"] / H_NULL > 0.9)},
    {"analysis": "4. joint / signatures", "space": "15 signatures", "fit": "joint",
     "K": K_opt, "ratio_soft": mix["ratio_soft"], "ratio_hard": mix["ratio_hard"],
     "shared_frac": np.mean(mix["h_hard"] / H_NULL > 0.9)},
])
print("=== Joint fits: marker vs signature space ===")
print(summary_2x2.round(3).to_string(index=False))
print(f"\ncount-based null = {H_NULL:.4f} bits for both")

print("\n=== Per-archetype mixing, joint signature space ===")
t = ident[["arch", "label", "n_cells", "pct", "ratio_hard", "top_line", "top_line_pct", "verdict"]]
print(t.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(2); w = 0.35
axes[0].bar(x - w/2, summary_2x2["ratio_soft"], w, label="soft (summed W)", color="#377eb8",
            edgecolor="black")
axes[0].bar(x + w/2, summary_2x2["ratio_hard"], w, label="hard (argmax)", color="#ff7f00",
            edgecolor="black")
axes[0].axhline(1.0, color="black", lw=1.4, ls="--", label="perfect mixing")
axes[0].set_xticks(x); axes[0].set_xticklabels(["joint\nMARKERS", "joint\nSIGNATURES"])
axes[0].set_ylabel("H / H_null"); axes[0].set_ylim(0, 1.1)
axes[0].set_title("Cell-line mixing: does scoring programmes first\nyield more shared archetypes?",
                  fontweight="bold")
axes[0].legend(fontsize=9); axes[0].grid(True, axis="y", ls="--", alpha=0.4)

srt = ident.sort_values("ratio_hard")
cols = ["#d73027" if v == "line-preferential" else "#1a9850" for v in srt["verdict"]]
axes[1].barh(range(len(srt)), srt["ratio_hard"], color=cols, edgecolor="black")
axes[1].axvline(0.9, color="black", ls=":", lw=1.2, label="shared threshold")
axes[1].set_yticks(range(len(srt)))
axes[1].set_yticklabels([f"{r.arch}: {r.label[:30]}" for r in srt.itertuples()], fontsize=8)
axes[1].set_xlabel("H_hard / H_null")
axes[1].set_title("Per-archetype mixing\n(green = shared, red = line-preferential)",
                  fontweight="bold")
axes[1].legend(fontsize=9); axes[1].grid(True, axis="x", ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS / "PermCellJoint_vs_Marker.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Summary

In [ ]:
print("=" * 78)
print("Joint archetypes over PermCell signature scores -- summary")
print("=" * 78)
print(f"\n-- Input --")
print(f"  cells                : {X_scaled.shape[0]:,}")
print(f"  signatures           : {X_scaled.shape[1]} (raw unsmoothed permutation Z)")
print(f"  scores reused        : valid, PermCell scoring is per-cell (max deviation {worst:.1e})")

print(f"\n-- Arms --")
print(f"  Arm B lower at       : {(piv['gain_%'] > 0).sum()}/{len(piv)} K values, "
      f"median {piv['gain_%'].median():.1f}%")
print(f"  -> per-line PROGRAMME baselines persist after PermCell's row-centring")

print(f"\n-- Selection --")
print(f"  elbow K              : {K_ELBOW}")
print(f"  selected K           : {K_SEL}   (largest K <= elbow with all seeds converged)")
print(f"  cross-seed stability : {float(sel_df.loc[sel_df.K==K_SEL,'stability'].iloc[0]):.3f}")
print(f"  representative seed  : {MEDOID}")

print(f"\n-- Mixing (null = {H_NULL:.4f} bits) --")
print(f"  soft   : {100*mix['ratio_soft']:.1f}% of null")
print(f"  hard   : {100*mix['ratio_hard']:.1f}% of null")
print(f"  shared : {n_shared}/{K_opt} archetypes")

print(f"\n-- vs joint MARKER space (analysis 1, K={Wjm.shape[1]}) --")
print(f"  marker    hard ratio {mix_jm['ratio_hard']:.3f}")
print(f"  signature hard ratio {mix['ratio_hard']:.3f}   "
      f"({'BETTER' if mix['ratio_hard'] > mix_jm['ratio_hard'] else 'WORSE'} line sharing)")
print("=" * 78)